# Experiment Catalog — SoftDecision Perturbed

Comprehensive table of every perturbed experiment, with hyperparameters extracted from `log.txt`.

Key columns: **σ**, **activation**, **σ schedule** (decay), **learning rate**, **n_samples**, **loss_type**, **val_frac**.

In [ ]:
import os, sys
import pandas as pd

os.chdir(os.path.expanduser(
    "/cluster/tufts/hugheslab/kheuto01/code/PredictiveCO-Benchmark"))

from rethink_exp.exp_helpers import catalog_all_experiments

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

df = catalog_all_experiments()
print(f"Total experiments: {len(df)}")
df.groupby("problem").size()

## Per-Problem Tables

Display unique hyperparameter configurations for each problem.
Columns: **σ**, **activation** (output_activation), **σ_schedule**, **σ_start→σ_end**, **n_samples**, **pred_loss_weight**, **loss_type**, **lr**, **val_frac**

In [ ]:
# ── Key columns for the catalog ────────────────────────────────────
DISPLAY_COLS = [
    "prefix", "wave", "sigma", "output_activation", "sigma_schedule",
    "sigma_start", "sigma_end", "n_samples", "pred_loss_weight",
    "loss_type", "lr", "val_frac",
]

for prob, gdf in df.groupby("problem"):
    print(f"\n{'='*100}")
    print(f"  {prob}  ({len(gdf)} experiments)")
    print(f"{'='*100}")
    cols = [c for c in DISPLAY_COLS if c in gdf.columns]
    display(gdf[cols].reset_index(drop=True))

## Unique Hyperparameter Combinations

Deduplicated view — one row per unique (σ, activation, schedule, n_samples, pred_loss_weight, loss_type) combo, with LRs collapsed.

In [ ]:
# ── Collapse learning rates into a list per unique config ──────────
GROUP_KEYS = [
    "problem", "sigma", "output_activation", "sigma_schedule",
    "sigma_start", "sigma_end", "n_samples", "pred_loss_weight",
    "loss_type", "val_frac",
]

def collapse_lr(g):
    lrs = sorted(g["lr"].dropna().unique())
    return pd.Series({
        "learning_rates": ", ".join(f"{x:g}" for x in lrs),
        "n_runs": len(g),
        "wave_range": f"{g['wave'].min()}-{g['wave'].max()}" if g["wave"].max() > 0 else "0",
    })

summary = df.groupby(GROUP_KEYS, dropna=False).apply(collapse_lr).reset_index()

for prob, gdf in summary.groupby("problem"):
    print(f"\n{'='*100}")
    print(f"  {prob}  ({gdf['n_runs'].sum()} runs, {len(gdf)} unique configs)")
    print(f"{'='*100}")
    show = gdf.drop(columns=["problem"]).sort_values(
        ["sigma", "output_activation", "sigma_schedule"]).reset_index(drop=True)
    display(show)

## LaTeX Export

Generate a LaTeX table (one per problem) suitable for inclusion in a paper.

In [ ]:
# ── LaTeX tables ───────────────────────────────────────────────────
LATEX_COLS = [
    "sigma", "output_activation", "sigma_schedule",
    "sigma_start", "sigma_end", "n_samples",
    "pred_loss_weight", "loss_type", "learning_rates",
    "val_frac", "n_runs",
]
LATEX_RENAME = {
    "sigma": r"$\sigma$",
    "output_activation": "Activation",
    "sigma_schedule": r"$\sigma$ Schedule",
    "sigma_start": r"$\sigma_{\rm start}$",
    "sigma_end": r"$\sigma_{\rm end}$",
    "n_samples": "$N$",
    "pred_loss_weight": r"$\lambda_{\rm pred}$",
    "loss_type": "Loss",
    "learning_rates": "LR(s)",
    "val_frac": "Val Frac",
    "n_runs": "Runs",
}

for prob, gdf in summary.groupby("problem"):
    show = (gdf[LATEX_COLS]
            .sort_values(["sigma", "output_activation", "sigma_schedule"])
            .reset_index(drop=True)
            .fillna("—")
            .rename(columns=LATEX_RENAME))
    latex = show.to_latex(index=False, escape=False,
                          caption=f"SoftDecision Perturbed configs — {prob}",
                          label=f"tab:perturb_{prob.replace('-','_')}")
    print(f"% ── {prob} ──")
    print(latex)
    print()

In [ ]:
# ── Save full catalog to CSV ──────────────────────────────────────
out_path = "rethink_exp/experiment_catalog.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows → {out_path}")